<a href="https://colab.research.google.com/github/chryyy/LiteMedSAM-LoRA-Coronary-Segmentation/blob/main/%E2%AD%90%EF%B8%8F_LiteMedSam_Finale_(Dice_0%2C8747)_%E2%AD%90%EF%B8%8F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🫀 LiteMedSAM LoRA per Segmentazione Coronarica 3D via Slicing 2.5D

 Segmentazione automatica delle arterie coronarie su volumi CT (ASOCA Dataset) utilizzando **LiteMedSAM**, una versione leggera ed efficiente del modello *Segment Anything* (SAM) adattata per l'imaging medico.

**Metodologia e Implementazione**
* **Architettura:** TinyViT-5M (Image Encoder) + Prompt Encoder + Mask Decoder.
* **Strategia:** Approccio "2.5D". Il volume 3D (.nrrd) viene decomposto in slice 2D, processato singolarmente e ricostruito.
* **Preprocessing:** Windowing Hounsfield (-200, 500 HU), Resize a 224x224 (con patch dinamico per compatibilità pesi) e Augmentation (Flip/Rotate/Noise).
* **Training:** Fine-tuning con tecnica **LoRA (Low-Rank Adaptation)** e congelamento del backbone per preservare i pesi pre-addestrati.
* **Loss Function:** Loss ibrida combinata (50% Cross Entropy + 50% Dice Loss) per massimizzare la precisione sui bordi.
* **Post-Processing:** Pipeline "Sandwich" morfologica (rimozione piccoli oggetti, chiusura gap, filtro isole) per pulire le maschere.
* **Output:** Generazione automatica di mesh 3D (.stl) pronte per la stampa o visualizzazione CAD.

## ⚙️ Setup

### Importa modello

In [ ]:
# === CELLA 1: Installazione e Configurazione ===

import os
import sys

!pip install numpy-stl

# 1. Installiamo le dipendenze (aggiunto -q per pulizia)
print("Installazione dipendenze in corso...")
!pip install -q SimpleITK opencv-python matplotlib
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

# 2. Cloniamo LiteMedSAM-LoRA
repo_name = "LiteMedSAM-LoRA"
if not os.path.exists(repo_name):
    print(f"Clonazione di {repo_name}...")
    !git clone https://github.com/lseventeen/LiteMedSAM-LoRA.git
else:
    print(f"Repository {repo_name} già presente.")

# 3. Setup ambiente specifico
# Salviamo la directory corrente per poterci tornare se serve
base_dir = os.getcwd()
repo_dir = os.path.join(base_dir, repo_name)

if os.getcwd() != repo_dir:
    os.chdir(repo_dir)
    print(f"Entrato in: {os.getcwd()}")
    !pip install -q -e .

# Aggiungiamo al path
if repo_dir not in sys.path:
    sys.path.append(repo_dir)

# 4. Importazioni
import numpy as np
import torch
import SimpleITK as sitk
import cv2
import matplotlib.pyplot as plt
from segment_anything import sam_model_registry

# 5. Verifica GPU (CRUCIALE per i tempi di calcolo)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Ambiente configurato. Device in uso: {device.upper()}")

### Importa pesi

In [ ]:
# === CELLA 2 (DEFINITIVA): Setup Percorsi e Drive ===
import os
import glob
from google.colab import drive

# 1. Collegamento Google Drive
if not os.path.exists('/content/drive'):
    print("📂 Sto montando Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive già collegato.")

# 2. Configurazione Percorsi
# Cartella base dei pesi su Drive
base_drive_folder = "/content/drive/MyDrive/Progetto Neuroengineering/Pesi LiteMedSam"
trained_drive_folder = os.path.join(base_drive_folder, "Pesi LiteMedSam (Trainato)")

# Definiamo i percorsi completi dei file
path_base_weights = os.path.join(base_drive_folder, "lite_medsam.pth")
path_best_weights = os.path.join(trained_drive_folder, "LiteMedSAM_best.pth")

# 3. Impostiamo le Variabili d'Ambiente (così la Cella 6 le può leggere)
os.environ['PATH_BASE_WEIGHTS'] = path_base_weights
os.environ['PATH_BEST_WEIGHTS'] = path_best_weights

# 4. Report Stato
print("\n🔍 Stato File Modello:")
if os.path.exists(path_best_weights):
    print(f"   💎 Trovato modello addestrato (BEST): {os.path.basename(path_best_weights)}")
    print("      -> Sarà usato come prioritario.")
else:
    print(f"   ⚪ Nessun modello addestrato trovato in: {trained_drive_folder}")

if os.path.exists(path_base_weights):
    print(f"   📦 Trovato modello base (BASE): {os.path.basename(path_base_weights)}")
else:
    print(f"   ❌ ATTENZIONE: Manca il modello base 'lite_medsam.pth'!")

print("\n✅ Configurazione percorsi completata.")

## 🩻 Dataset

### ⚡️ Furbata: dataset su content per max velocità

In [ ]:
# === CELLA EXTRA: Boost Velocità (Copia da Drive a /content) ===
import shutil
import os
import time

# 1. Definiamo i percorsi
drive_dataset_path = "/content/drive/MyDrive/Progetto Neuroengineering/DATASET ASOCA"
content_dataset_path = "/content/DATASET_ASOCA_FAST" # Nome chiaro per la copia veloce

# 2. Copia Intelligente
if not os.path.exists(content_dataset_path):
    print(f"🚀 Avvio copia del dataset nella memoria veloce (/content)...")
    print(f"   Sorgente (Drive): {drive_dataset_path}")
    print(f"   Destinazione (Content): {content_dataset_path}")

    start_time = time.time()
    try:
        # copytree clona l'intera cartella ricorsivamente
        shutil.copytree(drive_dataset_path, content_dataset_path)
        elapsed = time.time() - start_time
        print(f"✅ Copia in /content completata in {elapsed:.1f} secondi!")
        print("   Ora il training leggerà i dati dall'SSD locale (Fulmineo). ⚡")

        # Salviamo il percorso nella variabile d'ambiente per il Blocco 3
        os.environ['ASOCA_DATA_PATH'] = content_dataset_path

    except Exception as e:
        print(f"❌ Errore durante la copia: {e}")
        print("   Userò il percorso lento su Drive come fallback.")
        os.environ['ASOCA_DATA_PATH'] = drive_dataset_path
else:
    print("✅ Dataset già presente in /content. Skip copia.")
    os.environ['ASOCA_DATA_PATH'] = content_dataset_path

### Configurazione, split, visualizzazione

In [ ]:
# === CELLA 3 (MODIFICATA PER CONTENT): Configurazione ===
import os
import glob
import random
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt

class MockConfig:
    class random:
        seed = 42
    class dataset:
        class ASOCA:

            # Cerchiamo il percorso veloce impostato dalla Cella Extra.
            # Se non c'è, usiamo Drive come fallback.
            default_drive = "/content/drive/MyDrive/Progetto Neuroengineering/DATASET ASOCA"

            # Qui avviene la magia: os.environ.get legge "/content/..." se la copia è riuscita
            BASE_PATH = os.environ.get('ASOCA_DATA_PATH', default_drive)

            print(f"📂 Configurazione Dataset impostata su: {BASE_PATH}")

            path_normal = os.path.join(BASE_PATH, 'Normal')
            path_diseased = os.path.join(BASE_PATH, 'Diseased')

            # Dizionario per i dati
            split = {'train': [], 'val': [], 'test': []}

config = MockConfig()

# --- 2. FUNZIONE DI SCANSIONE E SPLIT ---
def populate_and_split_files(cfg_obj, split_ratio=0.8):
    print("🔎 Scansione cartelle 'CTCA' in corso...")

    # Pattern di ricerca
    patterns = [
        os.path.join(cfg_obj.dataset.ASOCA.path_normal, "CTCA", "*.nrrd"),
        os.path.join(cfg_obj.dataset.ASOCA.path_diseased, "CTCA", "*.nrrd")
    ]

    all_files = []
    for p in patterns:
        files = glob.glob(p)
        all_files.extend(files)
        print(f"   Trovati {len(files)} file in: {os.path.dirname(p)}")

    if not all_files:
        print("❌ ATTENZIONE: Nessun file trovato! Controlla i percorsi su Drive.")
        return cfg_obj

    # Mescoliamo (Shuffle) per evitare bias
    random.seed(cfg_obj.random.seed)
    random.shuffle(all_files)

    # Split 80/20
    split_idx = int(len(all_files) * split_ratio)
    train_files = all_files[:split_idx]
    val_files = all_files[split_idx:]

    cfg_obj.dataset.ASOCA.split['train'] = train_files
    cfg_obj.dataset.ASOCA.split['val'] = val_files

    print(f"✅ Dataset diviso: {len(train_files)} Train, {len(val_files)} Validation (Totale: {len(all_files)})")
    return cfg_obj

config = populate_and_split_files(config)

# --- 3. VISUALIZZAZIONE DI PROVA ---
def visualizza_overlay(image_path):
    mask_path = image_path.replace("CTCA", "Annotations")
    filename = os.path.basename(image_path)

    if not os.path.exists(mask_path):
        print(f"❌ Annotazione non trovata per: {filename}")
        return

    try:
        img_data = sitk.GetArrayFromImage(sitk.ReadImage(image_path)) # (Z, Y, X)
        msk_data = sitk.GetArrayFromImage(sitk.ReadImage(mask_path))

        # Trova la slice con più "arteria"
        pixels_per_slice = np.sum(msk_data, axis=(1, 2))
        z_idx = np.argmax(pixels_per_slice) if np.max(pixels_per_slice) > 0 else img_data.shape[0] // 2

        plt.figure(figsize=(6, 6))
        plt.title(f"Check Dati: {filename} | Slice {z_idx}")
        # Show CT (Grigio)
        plt.imshow(img_data[z_idx], cmap='gray', vmin=-200, vmax=500)
        # Show Mask (Rosso semi-trasparente)
        mask_slice = msk_data[z_idx]
        plt.imshow(np.ma.masked_where(mask_slice == 0, mask_slice), cmap='autumn', alpha=0.6)
        plt.axis('off')
        plt.show()
    except Exception as e:
        print(f"❌ Errore visualizzazione: {e}")

# Test visuale sul primo file del training
if config.dataset.ASOCA.split['train']:
    visualizza_overlay(config.dataset.ASOCA.split['train'][0])

### Dataset Avanzato e DataLoaders

In [ ]:
# === CELLA 4 (ZOOM DINAMICO + 2.5D): Dataset Super-Risoluzione ===
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import numpy as np
from torch.utils.data import Dataset, DataLoader
import SimpleITK as sitk
import os
import random

class DatasetLiteMEDSAM(Dataset):
    def __init__(self, file_list, img_size=224, bbox_shift=5, do_augment=True):
        self.file_list = file_list
        self.target_size = img_size
        self.bbox_shift = bbox_shift
        self.do_augment = do_augment
        # Dimensione del "ritaglio" zoomato (più è piccolo, più zoomiamo)
        self.crop_size = 256

    def __len__(self):
        return len(self.file_list)

    def load_volume(self, path):
        sitk_img = sitk.ReadImage(path)
        img_arr = sitk.GetArrayFromImage(sitk_img)
        # Normalizzazione
        img_arr = np.clip(img_arr, -200, 500)
        img_arr = (img_arr + 200) / 700.0
        return torch.from_numpy(img_arr).float()

    def load_label(self, img_path):
        label_path = img_path.replace("CTCA", "Annotations")
        if not os.path.exists(label_path): return None
        sitk_lbl = sitk.ReadImage(label_path)
        return torch.from_numpy(sitk.GetArrayFromImage(sitk_lbl)).long()

    # --- 2.5D Context ---
    def get_25d_slice(self, volume, z):
        max_z = volume.shape[0] - 1
        z_prev = max(0, z - 1)
        z_next = min(max_z, z + 1)
        return torch.stack([volume[z_prev], volume[z], volume[z_next]], dim=0)

    # --- ZOOM DINAMICO (CROP) ---
    def crop_zoom(self, img_3c, label_2d):
        """
        Ritaglia un quadrato attorno all'arteria per mantenere alta risoluzione.
        """
        _, h, w = img_3c.shape

        # Troviamo dove è l'arteria (bounding box della label)
        y_ind, x_ind = np.where(label_2d.numpy() > 0)

        if len(y_ind) > 0:
            # Centro dell'arteria
            cy = (np.min(y_ind) + np.max(y_ind)) // 2
            cx = (np.min(x_ind) + np.max(x_ind)) // 2

            # Aggiungiamo un offset casuale al centro (per robustezza)
            if self.do_augment:
                jitter = 20
                cy += random.randint(-jitter, jitter)
                cx += random.randint(-jitter, jitter)
        else:
            # Se la slice è vuota, centro dell'immagine
            cy, cx = h // 2, w // 2

        # Calcoliamo i bordi del ritaglio
        half_size = self.crop_size // 2
        min_y = max(0, cy - half_size)
        max_y = min(h, cy + half_size)
        min_x = max(0, cx - half_size)
        max_x = min(w, cx + half_size)

        # Eseguiamo il ritaglio (Crop)
        cropped_img = img_3c[:, min_y:max_y, min_x:max_x]
        cropped_lbl = label_2d[min_y:max_y, min_x:max_x]

        return cropped_img, cropped_lbl

    def resize_and_get_box(self, img, label):
        # Resize finale a 224x224 (Input del modello)
        # Nota: Se il crop era 256, stiamo riducendo pochissimo -> Alta Qualità!
        img = img.unsqueeze(0) # (1, 3, H, W)
        label = label.unsqueeze(0).unsqueeze(0) # (1, 1, H, W)

        img = F.interpolate(img, size=(self.target_size, self.target_size), mode='bilinear', align_corners=False)
        label = F.interpolate(label.float(), size=(self.target_size, self.target_size), mode='nearest').long()

        img = img.squeeze(0)
        label = label.squeeze() # (224, 224)

        # Calcolo BBox (Prompt) sulla label zoomata
        lbl_np = label.numpy()
        y_ind, x_ind = np.where(lbl_np > 0)

        if len(y_ind) > 0:
            pad = self.bbox_shift
            x_min = max(0, np.min(x_ind) - np.random.randint(0, pad))
            x_max = min(self.target_size, np.max(x_ind) + np.random.randint(0, pad))
            y_min = max(0, np.min(y_ind) - np.random.randint(0, pad))
            y_max = min(self.target_size, np.max(y_ind) + np.random.randint(0, pad))
            bbox = np.array([x_min, y_min, x_max, y_max])
        else:
            bbox = np.array([0, 0, self.target_size, self.target_size])

        return img, label, bbox

    def apply_augmentation(self, img, label):
        # Augmentation geometrica (Flip/Rotate)
        if torch.rand(1) > 0.5:
            img = TF.hflip(img)
            label = TF.hflip(label)
        if torch.rand(1) > 0.5:
            img = TF.vflip(img)
            label = TF.vflip(label)
        k = torch.randint(0, 4, (1,)).item()
        if k > 0:
            img = torch.rot90(img, k, dims=[-2, -1])
            label = torch.rot90(label, k, dims=[-2, -1])
        return img, label

    def __getitem__(self, idx):
        path = self.file_list[idx]
        try:
            vol = self.load_volume(path)
            lbl = self.load_label(path)
        except: return self.__getitem__((idx + 1) % len(self.file_list))
        if lbl is None: return self.__getitem__((idx + 1) % len(self.file_list))

        # Sampling
        valid = torch.where(lbl.sum(dim=(1, 2)) > 10)[0]
        z = np.random.choice(valid.numpy()) if len(valid) > 0 else vol.shape[0]//2

        # 1. Estrazione 2.5D
        img_3c = self.get_25d_slice(vol, z)
        label_2d = lbl[z]

        # 2. ZOOM DINAMICO (Crop mirato sull'arteria)
        img_crop, label_crop = self.crop_zoom(img_3c, label_2d)

        # 3. Augmentation
        if self.do_augment:
            img_crop, label_crop = self.apply_augmentation(img_crop, label_crop)

        # 4. Resize finale e Box
        final_img, final_lbl, bbox = self.resize_and_get_box(img_crop, label_crop)

        return {"image": final_img, "label": final_lbl, "bbox": bbox, "original_path": path}

# --- ISTANZIAZIONE ---
print("⚙️ Configurazione Dataset ZOOM 2.5D...")
if 'config' in locals() and len(config.dataset.ASOCA.split['train']) > 0:
    dataset_litemedsam_aug = DatasetLiteMEDSAM(config.dataset.ASOCA.split['train'], do_augment=True)
    train_loader = DataLoader(dataset_litemedsam_aug, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)

    val_files = config.dataset.ASOCA.split['val']
    dataset_val_noaug = DatasetLiteMEDSAM(val_files, do_augment=False)
    val_loader = DataLoader(dataset_val_noaug, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
    print("✅ Dataset pronti!")

### 🔍 Augmentation e Bounding Box

In [ ]:
# === CELLA 5 - DI VERIFICA (VISUALIZZAZIONE AUGMENTATION) ===
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualizza_batch_augmentation(loader):
    # 1. Prendiamo un singolo batch dal DataLoader
    batch = next(iter(loader))

    images = batch['image'] # (B, 3, 256, 256)
    labels = batch['label'] # (B, 256, 256)
    bboxes = batch['bbox']  # (B, 4)
    paths = batch['original_path']

    batch_size = images.shape[0]

    plt.figure(figsize=(16, 5))
    plt.suptitle("Verifica Augmentation: Immagine + Maschera (Rosso) + BBox (Verde)", fontsize=16)

    for i in range(batch_size):
        ax = plt.subplot(1, batch_size, i + 1)

        # --- Preparazione Immagine ---
        # Convertiamo da Tensore (C, H, W) a Numpy (H, W, C) per matplotlib
        img_disp = images[i].permute(1, 2, 0).cpu().numpy()

        # Normalizzazione visiva: Se i valori sono fuori da 0-1 (es. -1 a 1), li riportiamo a 0-1
        # Questo serve solo per vederla bene a video, non cambia i dati del training
        img_disp = (img_disp - img_disp.min()) / (img_disp.max() - img_disp.min() + 1e-5)

        plt.imshow(img_disp)

        # --- Sovrapposizione Maschera ---
        mask_disp = labels[i].cpu().numpy()
        # Usiamo 'masked_where' per rendere trasparenti i pixel di sfondo (valore 0)
        mask_overlay = np.ma.masked_where(mask_disp == 0, mask_disp)
        plt.imshow(mask_overlay, cmap='autumn', alpha=0.5) # Rosso semi-trasparente

        # --- Disegno Bounding Box ---
        # Il box è [x_min, y_min, x_max, y_max]
        x1, y1, x2, y2 = bboxes[i].numpy()
        width = x2 - x1
        height = y2 - y1

        # Crea il rettangolo
        rect = patches.Rectangle((x1, y1), width, height, linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)

        # Titolo con info file
        file_name = os.path.basename(paths[i])
        # Accorciamo il nome se troppo lungo
        short_name = file_name if len(file_name) < 15 else "..." + file_name[-15:]
        plt.title(f"File: {short_name}\nSlice random")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# --- ESEGUI LA VERIFICA ---
print("👁️ Visualizzazione di un batch di training (con Augmentation)...")
# Nota: Eseguendo questa cella più volte vedrai risultati diversi perché l'augmentation è casuale!
visualizza_batch_augmentation(train_loader)

In [ ]:
# === CELLA 6 (DEFINITIVA): Costruzione Modello e Caricamento Pesi ===
import torch
import torch.nn as nn
import sys
import os
import argparse

repo_path = "/content/LiteMedSAM-LoRA"
if repo_path not in sys.path: sys.path.insert(0, repo_path)

try:
    from models.ImageEncoder.tinyvit.tiny_vit import TinyViT
    from segment_anything.modeling import MaskDecoder, PromptEncoder, TwoWayTransformer
except ImportError as e: raise ImportError(f"❌ Errore Import: {e}")

class MedSAM_Lite(nn.Module):
    def __init__(self, image_encoder, mask_decoder, prompt_encoder):
        super().__init__()
        self.image_encoder = image_encoder
        self.mask_decoder = mask_decoder
        self.prompt_encoder = prompt_encoder

    def forward(self, image, box):
        image_embedding = self.image_encoder(image)
        if box.dim() == 2:
            box = box.unsqueeze(1)
        sparse_embeddings, dense_embeddings = self.prompt_encoder(
            points=None, boxes=box, masks=None,
        )
        low_res_masks, iou_predictions = self.mask_decoder(
            image_embeddings=image_embedding,
            image_pe=self.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_embeddings,
            dense_prompt_embeddings=dense_embeddings,
            multimask_output=False,
        )
        return low_res_masks, iou_predictions

def build_litemedsam_224(checkpoint_path, device):
    print(f"🏗️  Costruzione architettura LiteMedSAM (TinyViT-5M)...")

    # Configurazione TinyViT + LoRA
    args = argparse.Namespace(mod='sam_lora', mid_dim=None, out_indices=[0, 1, 2, 3])
    kwargs = {
        'img_size': 224, 'in_chans': 3,
        'embed_dims': [64, 128, 160, 320], 'depths': [2, 2, 6, 2],
        'num_heads': [2, 4, 5, 10], 'window_sizes': [7, 7, 14, 7],
        'mlp_ratio': 4.0, 'drop_path_rate': 0.0, 'use_checkpoint': False,
        'mbconv_expand_ratio': 4.0, 'local_conv_size': 3,
    }

    try:
        medsam_lite_image_encoder = TinyViT(args, **kwargs)
    except TypeError:
        medsam_lite_image_encoder = TinyViT(args, embed_dims=[64, 128, 160, 320], window_sizes=[7, 7, 14, 7])

    if medsam_lite_image_encoder.patch_embed.seq[0].c.out_channels != 32:
        raise ValueError("❌ ERRORE FATALE: Dimensioni modello errate.")

    medsam_lite_prompt_encoder = PromptEncoder(
        embed_dim=256, image_embedding_size=(64, 64), input_image_size=(224, 224), mask_in_chans=16
    )

    medsam_lite_mask_decoder = MaskDecoder(
        num_multimask_outputs=3,
        transformer=TwoWayTransformer(depth=2, embedding_dim=256, mlp_dim=2048, num_heads=8),
        transformer_dim=256,
        iou_head_depth=3,
        iou_head_hidden_dim=256,
    )

    model = MedSAM_Lite(
        image_encoder=medsam_lite_image_encoder,
        mask_decoder=medsam_lite_mask_decoder,
        prompt_encoder=medsam_lite_prompt_encoder
    )

    # Caricamento Pesi
    if checkpoint_path and os.path.exists(checkpoint_path):
        print(f"🔄 Caricamento pesi da: {os.path.basename(checkpoint_path)}")
        state_dict = torch.load(checkpoint_path, map_location='cpu')
        keys = model.load_state_dict(state_dict, strict=False)

        missing_lora = [k for k in keys.missing_keys if "lora" in k]
        if len(missing_lora) > 0:
            print(f"ℹ️  Modalità: Pre-Training (Mancano {len(missing_lora)} layer LoRA da addestrare)")
        else:
            print("✅ Modalità: Fine-Tuned (Modello completo e pronto!)")
    else:
        print(f"❌ ERRORE: File pesi non trovato: {checkpoint_path}")

    return model.to(device)

# --- RE-INIZIALIZZAZIONE DATASET (Per sicurezza) ---
if 'config' in locals() and len(config.dataset.ASOCA.split['train']) > 0:
    print("\n🔄 Verifica Dataset...")
    dataset_litemedsam_aug = DatasetLiteMEDSAM(config.dataset.ASOCA.split['train'], img_size=224, do_augment=True)
    train_loader = DataLoader(dataset_litemedsam_aug, batch_size=4, shuffle=True)

    # Se il validation set è vuoto, usiamo il train per evitare crash
    val_files = config.dataset.ASOCA.split['val'] if len(config.dataset.ASOCA.split['val']) > 0 else config.dataset.ASOCA.split['train']
    dataset_val_noaug = DatasetLiteMEDSAM(val_files, img_size=224, do_augment=False)
    val_loader = DataLoader(dataset_val_noaug, batch_size=4, shuffle=False)
    print("✅ Dataset 224px configurati.")

# --- SELEZIONE PESI AUTOMATICA ---
device = "cuda" if torch.cuda.is_available() else "cpu"

# Recuperiamo i percorsi impostati nella Cella 2
path_best = os.environ.get('PATH_BEST_WEIGHTS')
path_base = os.environ.get('PATH_BASE_WEIGHTS')

# Logica di priorità
ckpt_to_load = None
if path_best and os.path.exists(path_best):
    print(f"\n💎 SCELTA: Uso i tuoi pesi addestrati (Best).")
    ckpt_to_load = path_best
elif path_base and os.path.exists(path_base):
    print(f"\n📦 SCELTA: Uso i pesi base (Pre-trained).")
    ckpt_to_load = path_base
else:
    print("\n❌ ERRORE CRITICO: Nessun file pesi disponibile su Drive o Variabili non settate.")

# Costruzione Finale
if ckpt_to_load:
    medsam_model = build_litemedsam_224(ckpt_to_load, device)
    # Fix obbligatorio per 224
    medsam_model.prompt_encoder.image_embedding_size = (56, 56)
    print(f"✅ Modello pronto su {device}!")

## 🏋🏻‍♀️ Training

### Pulizia memoria pre-training

In [ ]:
# === CELLA DI EMERGENZA: PULIZIA MEMORIA ===
import torch
import gc

# Svuota la cache della GPU
torch.cuda.empty_cache()
gc.collect()

print("🧹 Memoria GPU ripulita. Ora puoi lanciare il Training.")

### Addestramento

In [ ]:
# === CELLA 7 (DEFINITIVA - TVERSKY LOSS): Fix Modello & Training Loop ===
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import time
import types
import math
import os

# --- 1. MONKEY PATCH (Fix Bug Dimensionale 64 vs 56) ---
def forward_features_dynamic(self, x):
    x = self.patch_embed(x)
    for layer in self.layers:
        x = layer(x)

    B, N, C = x.size()
    size = int(math.sqrt(N))
    x = x.view(B, size, size, C)

    x = x.permute(0, 3, 1, 2)
    if hasattr(self, 'neck'):
        x = self.neck(x)
    return x

# Applichiamo il fix al modello in memoria
print("🔧 Applicazione patch dinamica al modello...")
medsam_model.image_encoder.forward_features = types.MethodType(forward_features_dynamic, medsam_model.image_encoder)
medsam_model.prompt_encoder.image_embedding_size = (56, 56)
print("✅ Patch applicata correttamente.")

# --- 2. CONFIGURAZIONE TRAINING ---
MAX_STEPS = 1000
EVAL_INTERVAL = 100
BATCH_SIZE = 4          # Usiamo il batch size di Colab Pro
ACCUMULATION_STEPS = 1  # Accumulo per compensare
LR = 0.0001
WD = 0.05

# --- 3. CONGELAMENTO PESI (Freezing) ---
print("\n🔒 Congelamento Backbone...")
trainable_params = []
for name, param in medsam_model.named_parameters():
    if any(x in name for x in ["lora", "mask_decoder", "prompt_encoder", "bias"]):
        param.requires_grad = True
        trainable_params.append(param)
    else:
        param.requires_grad = False

print(f"✅ Parametri addestrabili: {len(trainable_params)} tensor groups.")
optimizer = optim.AdamW(trainable_params, lr=LR, weight_decay=WD)

# --- 4. NUOVA LOSS FUNCTION ASIMMETRICA (Tversky) ---
class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, smooth=1e-5):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth

    def forward(self, pred, target):
        # target qui è già [B, 1, H, W]
        pred = torch.sigmoid(pred)

        # Calcolo Intersection, FP, FN
        intersection = (pred * target).sum(dim=(2, 3))
        fp = ((1 - target) * pred).sum(dim=(2, 3))
        fn = (target * (1 - pred)).sum(dim=(2, 3))

        # Formula Tversky (beta alto penalizza FN)
        tversky = (intersection + self.smooth) / (intersection + self.alpha * fp + self.beta * fn + self.smooth)
        return (1.0 - tversky).mean()

class MedSAMLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.BCEWithLogitsLoss()
        self.tversky = TverskyLoss()

    def forward(self, pred, target):
        # 🟢 CORREZIONE DEL DIMENSIONAMENTO: Espande [B, H, W] in [B, 1, H, W]
        target = target.unsqueeze(1).float()

        loss_ce = self.ce(pred, target) # Ora target e pred hanno lo stesso size!
        loss_tversky = self.tversky(pred, target)

        # Blend 50% BCE e 50% Tversky
        return 0.5 * loss_ce + 0.5 * loss_tversky

loss_fn = MedSAMLoss()

# --- 5. TRAINING LOOP (Step-Based) ---
if 'dataset_litemedsam_aug' not in globals(): raise NameError("❌ Dataset non definito!")

# Rigeneriamo i loader per sicurezza
train_loader = DataLoader(dataset_litemedsam_aug, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(dataset_val_noaug, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"\n🚀 Inizio Training: {MAX_STEPS} Steps con Tversky Loss...")
device = "cuda" if torch.cuda.is_available() else "cpu"
medsam_model.to(device)
medsam_model.train()

global_step = 0
best_val_loss = float('inf')
start_time = time.time()
train_loss_accum = 0.0

while global_step < MAX_STEPS:
    for batch in train_loader:
        if global_step >= MAX_STEPS: break

        # Forward
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        bboxes = batch['bbox'].float().to(device)

        pred_masks_raw, _ = medsam_model(images, bboxes)
        pred_masks = F.interpolate(pred_masks_raw, size=(224, 224), mode="bilinear", align_corners=False)

        # Loss & Backward
        loss = loss_fn(pred_masks, labels) # ACCUMULATION_STEPS è 1
        loss.backward()
        train_loss_accum += loss.item()

        # Optimizer Step
        if (global_step + 1) % ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()
            optimizer.zero_grad()

            # Log
            actual_step = (global_step + 1) // ACCUMULATION_STEPS
            if actual_step % 10 == 0:
                elapsed = time.time() - start_time
                print(f"Step {actual_step}/{MAX_STEPS//ACCUMULATION_STEPS} | Train Loss: {train_loss_accum:.4f} | Time: {elapsed:.0f}s")
                train_loss_accum = 0.0

            # Validation
            if actual_step % (EVAL_INTERVAL // ACCUMULATION_STEPS) == 0:
                print(f"\n🔍 Validazione Step {actual_step}...")
                medsam_model.eval()
                val_loss_tot = 0.0
                with torch.no_grad():
                    for v_batch in val_loader:
                        v_img = v_batch['image'].to(device)
                        v_lbl = v_batch['label'].to(device)
                        v_box = v_batch['bbox'].float().to(device)
                        v_p, _ = medsam_model(v_img, v_box)
                        v_p = F.interpolate(v_p, size=(224, 224), mode="bilinear", align_corners=False)
                        val_loss_tot += loss_fn(v_p, v_lbl).item()

                avg_val = val_loss_tot / len(val_loader)
                print(f"   📉 Val Loss: {avg_val:.4f} (Best: {best_val_loss:.4f})")

                if avg_val < best_val_loss:
                    best_val_loss = avg_val
                    save_path = os.environ.get('PATH_BEST_WEIGHTS', "LiteMedSAM_best.pth")
                    torch.save(medsam_model.state_dict(), save_path)
                    print(f"   🏆 Modello salvato in: {os.path.basename(save_path)}")

                medsam_model.train()

        global_step += 1

print("\n✅ Training Completato!")

### Copio i nuovi pesi (best e latest) nella cartella dedicata su Drive:

In [ ]:
# === CELLA DI SALVATAGGIO MANUALE (Intelligente) ===
import os
import torch
import datetime

# 1. Recuperiamo il percorso target su Drive (impostato nella Cella 2)
path_best_drive = os.environ.get('PATH_BEST_WEIGHTS')
dir_drive = os.path.dirname(path_best_drive)

print(f"📂 Verifica salvataggio su Drive: {dir_drive}")

# 2. Controllo Esistenza File 'Best'
if os.path.exists(path_best_drive):
    # Otteniamo l'orario di ultima modifica per conferma
    timestamp = os.path.getmtime(path_best_drive)
    time_str = datetime.datetime.fromtimestamp(timestamp).strftime('%H:%M:%S')
    print(f"✅ CONFERMATO: 'LiteMedSAM_best.pth' è presente su Drive.")
    print(f"   🕒 Ultimo aggiornamento: oggi alle {time_str}")
else:
    print(f"⚠️ ATTENZIONE: Il file 'LiteMedSAM_best.pth' NON è stato trovato su Drive.")

# 3. Opzione: Salvataggio Forzato dello stato CORRENTE (quello in memoria RAM)
# Utile se hai stoppato il training a mano e vuoi salvare dove sei arrivato
save_now = True # <--- Metti False se non vuoi salvare nulla manualmente

if save_now:
    # Nome diverso per non sovrascrivere il Best
    path_manual = os.path.join(dir_drive, "LiteMedSAM_manual_save.pth")
    print(f"\n💾 Eseguo salvataggio manuale dello stato corrente della memoria...")
    try:
        torch.save(medsam_model.state_dict(), path_manual)
        print(f"✅ Salvato con successo: {os.path.basename(path_manual)}")
    except Exception as e:
        print(f"❌ Errore durante il salvataggio manuale: {e}")

print("\n🏁 Procedura di backup terminata.")

## 🏆 Risultati e metriche

### Singola Slice

In [ ]:
# === BLOCCO 8: Visualizzazione Slice con Metriche (Dice/IoU) ===
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import torch
import torch.nn.functional as F
import os

# 1. Funzione Helper per calcolare le metriche
def calculate_metrics(pred_mask, gt_mask):
    # Appiattiamo i tensori (trasformiamo in vettori 1D)
    pred_flat = pred_mask.flatten()
    gt_flat = gt_mask.flatten()

    intersection = np.sum(pred_flat * gt_flat)
    union = np.sum(pred_flat) + np.sum(gt_flat)

    # Dice Score: 2 * Int / (Somma Pixel A + Somma Pixel B)
    dice = (2. * intersection + 1e-5) / (union + 1e-5)

    # IoU: Int / (Union - Int)
    iou = (intersection + 1e-5) / (union - intersection + 1e-5)

    return dice, iou

def show_prediction_with_metrics(model, dataset, index=0, device='cuda'):
    model.eval()
    sample = dataset[index]

    # Preparazione input
    image = sample['image'].to(device).unsqueeze(0)
    label = sample['label'].numpy() # Ground Truth originale
    bbox = torch.tensor(sample['bbox']).float().to(device).unsqueeze(0)
    if bbox.dim() == 2: bbox = bbox.unsqueeze(1)

    # Predizione
    with torch.no_grad():
        pred_logits, _ = model(image, bbox)
        # Resize per matchare la label (224x224)
        pred_logits = F.interpolate(pred_logits, size=(224, 224), mode="bilinear", align_corners=False)
        pred_mask = (torch.sigmoid(pred_logits) > 0.5).squeeze().cpu().numpy()

    # Calcolo Metriche
    dice, iou = calculate_metrics(pred_mask, label)

    # Preparazione Plot
    img_show = sample['image'].permute(1, 2, 0).numpy()
    img_show = (img_show - img_show.min()) / (img_show.max() - img_show.min())
    bbox_coords = sample['bbox']
    file_name = os.path.basename(sample['original_path'])

    # --- PLOT ---
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    plt.suptitle(f"File: {file_name[:20]}... | Dice: {dice:.4f} | IoU: {iou:.4f}", fontsize=16, color='navy')

    # A. Input
    axes[0].imshow(img_show)
    axes[0].set_title("Input + Prompt")
    rect = patches.Rectangle((bbox_coords[0], bbox_coords[1]), bbox_coords[2]-bbox_coords[0], bbox_coords[3]-bbox_coords[1], linewidth=2, edgecolor='lime', facecolor='none')
    axes[0].add_patch(rect)
    axes[0].axis('off')

    # B. Ground Truth
    axes[1].imshow(label, cmap='gray')
    axes[1].set_title("Ground Truth")
    axes[1].axis('off')

    # C. Predizione (Coloriamo errori in giallo)
    # Creiamo un'immagine RGB per evidenziare le differenze
    output_vis = np.zeros((224, 224, 3))
    output_vis[..., 1] = label      # Verde = Realtà
    output_vis[..., 0] = pred_mask  # Rosso = Predizione
    # Risultato:
    # Giallo = Corretto (Verde+Rosso)
    # Solo Verde = Falso Negativo (Mancato)
    # Solo Rosso = Falso Positivo (Inventato)

    axes[2].imshow(output_vis)
    axes[2].set_title("Sovrapposizione (Giallo=OK, Rosso=Err, Verde=Miss)")
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

# Test su un esempio random
print("📊 Analisi Singola Slice...")
idx = np.random.randint(0, len(val_loader.dataset))
show_prediction_with_metrics(medsam_model, val_loader.dataset, index=idx)

### Ricostruzione 3D delle coronarie

In [ ]:
# === BLOCCO 9 (FINALE - TTA): Inferenza Volumetrica Definitiva ===
import SimpleITK as sitk
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF # Nuovo import per TTA
import os
from skimage import measure
import plotly.graph_objects as go

# --- FUNZIONE HELPER: PREDIZIONE CON TTA ---
def predict_with_tta(model, input_tensor, bbox_tensor):
    """Esegue inferenza con TTA (Base, Flip H, Flip V) su un input 2.5D."""
    tta_preds = []

    # 1. Predizione Base (No Augmentation)
    logits, _ = model(input_tensor, bbox_tensor)
    tta_preds.append(torch.sigmoid(logits))

    # 2. Flip Orizzontale (H-Flip)
    input_flipped = TF.hflip(input_tensor)
    logits_flipped, _ = model(input_flipped, bbox_tensor)
    pred_flipped = torch.sigmoid(logits_flipped)
    tta_preds.append(TF.hflip(pred_flipped)) # Flipperizziamo la predizione indietro

    # 3. Flip Verticale (V-Flip)
    input_flipped = TF.vflip(input_tensor)
    logits_flipped, _ = model(input_flipped, bbox_tensor)
    pred_flipped = torch.sigmoid(logits_flipped)
    tta_preds.append(TF.vflip(pred_flipped))

    # Media delle predizioni (Soft Voting)
    avg_pred = torch.stack(tta_preds, dim=0).mean(dim=0)
    return avg_pred


# --- FUNZIONE PRINCIPALE DI INFERENZA (Adesso usa TTA) ---
def run_full_volume_inference(model, nrrd_path, device='cuda', crop_size=256):
    print(f"🔄 Caricamento volume (Mode: TTA + Zoom {crop_size}px + 2.5D)...")

    vol_img = sitk.ReadImage(nrrd_path)
    vol_arr = sitk.GetArrayFromImage(vol_img) # (Z, Y, X)

    mask_path = nrrd_path.replace("CTCA", "Annotations")
    if not os.path.exists(mask_path): return None, None
    mask_arr = sitk.GetArrayFromImage(sitk.ReadImage(mask_path))

    model.eval()
    prediction_3d = np.zeros_like(vol_arr)

    # Pre-processing Globale del volume (Norm)
    vol_norm = np.clip(vol_arr, -200, 500)
    vol_norm = (vol_norm + 200) / 700.0
    vol_tensor_full = torch.from_numpy(vol_norm).float()

    max_z, h_orig, w_orig = vol_arr.shape
    target_size = 224 # Modello LiteMedSAM

    with torch.no_grad():
        for z in range(max_z):
            label_slice = mask_arr[z, :, :]
            if np.sum(label_slice) == 0: continue

            # --- 1. PREPARAZIONE INPUT Z+XY ---
            ys, xs = np.where(label_slice > 0)
            cy, cx = (np.min(ys)+np.max(ys))//2, (np.min(xs)+np.max(xs))//2

            # Calcolo Coordinate del CROP (Zoom)
            half = crop_size // 2
            y1 = max(0, cy - half); y2 = min(h_orig, cy + half)
            x1 = max(0, cx - half); x2 = min(w_orig, cx + half)

            # Estrazione 2.5D
            z_prev, z_next = max(0, z-1), min(max_z-1, z+1)
            slice_3c = torch.stack([vol_tensor_full[z_prev], vol_tensor_full[z], vol_tensor_full[z_next]], dim=0)

            # Eseguiamo il CROP
            img_crop = slice_3c[:, y1:y2, x1:x2]

            # Resize per il modello (224x224)
            input_tensor = img_crop.unsqueeze(0)
            input_tensor = F.interpolate(input_tensor, size=(target_size, target_size), mode='bilinear', align_corners=False).to(device)

            # Prompt Box Scalato
            bbox = torch.tensor([[np.min(xs), np.min(ys), np.max(xs), np.max(ys)]]).float()
            bbox[:, 0] -= x1; bbox[:, 2] -= x1
            bbox[:, 1] -= y1; bbox[:, 3] -= y1
            scale_y = target_size / (y2 - y1); scale_x = target_size / (x2 - x1)
            bbox_scaled = bbox * torch.tensor([scale_x, scale_y, scale_x, scale_y])
            bbox_scaled = bbox_scaled.unsqueeze(0).to(device)

            # 7. Inferenza con TTA (Nuova riga)
            avg_logits = predict_with_tta(model, input_tensor, bbox_scaled)

            # 8. Resize Back e Incolla
            pred_crop = F.interpolate(avg_logits, size=(y2-y1, x2-x1), mode='bilinear', align_corners=False)
            pred_mask = (pred_crop > 0.5).squeeze().cpu().numpy()

            prediction_3d[z, y1:y2, x1:x2] = pred_mask

    return mask_arr, prediction_3d

# --- 2. VISUALIZZAZIONE MESH 3D (Uguale) ---
def visualize_coronaries_3d(mask_gt, mask_pred):
    print("🎨 Generazione Mesh 3D (Plotly)...")
    fig = go.Figure()

    if np.sum(mask_gt) > 0:
        verts, faces, _, _ = measure.marching_cubes(mask_gt, level=0.5)
        fig.add_trace(go.Mesh3d(x=verts[:,2], y=verts[:,1], z=verts[:,0], i=faces[:,0], j=faces[:,1], k=faces[:,2],
                                opacity=0.3, color='green', name='Ground Truth'))

    if np.sum(mask_pred) > 0:
        verts, faces, _, _ = measure.marching_cubes(mask_pred, level=0.5)
        fig.add_trace(go.Mesh3d(x=verts[:,2], y=verts[:,1], z=verts[:,0], i=faces[:,0], j=faces[:,1], k=faces[:,2],
                                opacity=1.0, color='red', name='Prediction (TTA)'))

    fig.update_layout(title="Ricostruzione 3D (TTA Finale)", scene=dict(aspectmode='data'))
    fig.show()

# --- 3. ESECUZIONE ---
if len(config.dataset.ASOCA.split['val']) > 0:
    test_file = config.dataset.ASOCA.split['val'][0]
    mask_true, mask_pred = run_full_volume_inference(medsam_model, test_file)

    if mask_true is not None:
        intersection = np.sum(mask_pred * mask_true)
        dice_3d = (2. * intersection) / (np.sum(mask_pred) + np.sum(mask_true) + 1e-5)
        print(f"\n🏆 NUOVISSIMO DICE SCORE 3D GLOBALE: {dice_3d:.4f}")
        visualize_coronaries_3d(mask_true, mask_pred)

### Generazione STL

In [ ]:
# === CELLA 10 (SMART UPDATE): Export STL solo se il modello è migliorato ===
import os
import numpy as np
import SimpleITK as sitk
from skimage import morphology, measure
from skimage.morphology import ball, binary_closing
from stl import mesh
import torch
import time

# --- 1. FUNZIONI DI SUPPORTO (Pulizia e STL) ---
def apply_sandwich_cleaning(segmentation_mask):
    mask_bool = segmentation_mask.astype(bool)
    mask_clean_1 = morphology.remove_small_objects(mask_bool, min_size=50)
    mask_closed = binary_closing(mask_clean_1, ball(2))
    mask_final = morphology.remove_small_objects(mask_closed, min_size=500)
    return mask_final.astype(np.uint8)

def save_stl(mask_3d, output_path, spacing=(1.0, 1.0, 1.0)):
    if np.sum(mask_3d) < 100:
        print(f"   ⚠️ Volume vuoto, salto STL: {os.path.basename(output_path)}")
        return
    try:
        verts, faces, _, _ = measure.marching_cubes(mask_3d, level=0.5)
        verts = verts * np.array(spacing[::-1])
        obj = mesh.Mesh(np.zeros(faces.shape[0], dtype=mesh.Mesh.dtype))
        for i, f in enumerate(faces):
            obj.vectors[i] = verts[f, :]
        obj.save(output_path)
        print(f"   💾 STL Aggiornato: {os.path.basename(output_path)}")
    except Exception as e:
        print(f"   ❌ Errore STL: {e}")

# --- 2. CONFIGURAZIONE AGGIORNAMENTO INTELLIGENTE ---
output_folder = "/content/drive/MyDrive/Progetto Neuroengineering/STL_Predictions"
os.makedirs(output_folder, exist_ok=True)

# Percorso del modello "Best"
model_path = os.environ.get('PATH_BEST_WEIGHTS')
if not model_path or not os.path.exists(model_path):
    print("⚠️ Attenzione: Non trovo il file dei pesi migliori. Rigenero tutto per sicurezza.")
    model_timestamp = float('inf') # Forza rigenerazione
else:
    model_timestamp = os.path.getmtime(model_path)
    print(f"🕒 Timestamp Modello Migliore: {time.ctime(model_timestamp)}")

# File da processare
files_to_process = config.dataset.ASOCA.split['train']

print(f"🚀 Controllo aggiornamenti per {len(files_to_process)} pazienti...")

for i, nrrd_path in enumerate(files_to_process):
    file_name = os.path.basename(nrrd_path).replace(".nrrd", ".stl")
    save_path = os.path.join(output_folder, file_name)

    # --- LOGICA SMART UPDATE ---
    should_process = True
    if os.path.exists(save_path):
        stl_timestamp = os.path.getmtime(save_path)
        # Se l'STL è più nuovo del modello, non serve rifarlo
        if stl_timestamp > model_timestamp:
            print(f"   ⏩ {file_name} è già aggiornato. Salto.")
            should_process = False

    if not should_process:
        continue

    print(f"\nProcessing {i+1}/{len(files_to_process)}: {file_name} (Nuova versione)...")

    try:
        # A. Caricamento e Spacing
        sitk_img = sitk.ReadImage(nrrd_path)
        spacing = sitk_img.GetSpacing()

        # B. Inferenza
        _, pred_raw = run_full_volume_inference(medsam_model, nrrd_path, device='cuda')

        if pred_raw is None: continue

        # C. Pulizia e Salvataggio
        pred_clean = apply_sandwich_cleaning(pred_raw)
        save_stl(pred_clean, save_path, spacing)

    except Exception as e:
        print(f"   ❌ Errore critico su {file_name}: {e}")

print("\n✅ Procedura di aggiornamento STL completata.")

In [ ]:
# =========================================================================
# BLOCCO 5 (FINALE): ESPORTAZIONE "GEMELLA" PER NETRON
# =========================================================================
import torch
import os
import argparse

# 1. Configurazione Percorso
save_folder = "."
if 'path_best' in globals() and path_best:
    save_folder = os.path.dirname(path_best)
elif 'args' in globals() and hasattr(args, 'resume') and args.resume:
    save_folder = args.resume

jit_filename = "litemedsam_netron_structure.pt"
save_path = os.path.join(save_folder, jit_filename)

print(f"📍 Creazione modello strutturale per Netron in: {save_path}")

# 2. COSTRUZIONE "MODELLO GEMELLO" (256x256)
# Costruiamo un modello identico ma configurato a 256 per aggirare i bug della libreria.
# Questo serve SOLO per la visualizzazione del grafico.

# Parametri compatibili con 256 (256 è divisibile per 8, non per 7)
args_netron = argparse.Namespace(mod='sam_lora', mid_dim=None, out_indices=[0, 1, 2, 3])
kwargs_netron = {
    'img_size': 256,         # <--- FORZIAMO 256
    'in_chans': 3,
    'embed_dims': [64, 128, 160, 320],
    'depths': [2, 2, 6, 2],
    'num_heads': [2, 4, 5, 10],
    'window_sizes': [8, 8, 16, 8], # <--- ADATTATO PER 256 (256/32 = 8)
    'mlp_ratio': 4.0,
    'drop_path_rate': 0.0,
    'use_checkpoint': False,
    'mbconv_expand_ratio': 4.0,
    'local_conv_size': 3,
}

try:
    print("🏗️  Istanzio TinyViT (versione 256px per export)...")
    # Istanziazione Encoder
    netron_image_encoder = TinyViT(args_netron, **kwargs_netron)

    # Istanziazione Prompt Encoder (Configurato per 256 -> embedding 64)
    netron_prompt_encoder = PromptEncoder(
        embed_dim=256,
        image_embedding_size=(64, 64), # 256 / 4 = 64
        input_image_size=(256, 256),
        mask_in_chans=16
    )

    # Istanziazione Mask Decoder (Identico)
    netron_mask_decoder = MaskDecoder(
        num_multimask_outputs=3,
        transformer=TwoWayTransformer(depth=2, embedding_dim=256, mlp_dim=2048, num_heads=8),
        transformer_dim=256,
        iou_head_depth=3,
        iou_head_hidden_dim=256,
    )

    # Assemblaggio
    netron_model = MedSAM_Lite(
        image_encoder=netron_image_encoder,
        mask_decoder=netron_mask_decoder,
        prompt_encoder=netron_prompt_encoder
    )

    netron_model.eval()

    # 3. TRACCIAMENTO
    # Creiamo input compatibili con questa versione
    dummy_img = torch.randn(1, 3, 256, 256)
    dummy_box = torch.tensor([[50.0, 50.0, 150.0, 150.0]])

    print(f"⏳ Tracciamento (JIT Tracing)...")
    with torch.no_grad():
        traced_netron = torch.jit.trace(netron_model, (dummy_img, dummy_box))
        torch.jit.save(traced_netron, save_path)

    print("\n" + "="*50)
    print(f"✅ FILE GENERATO CON SUCCESSO!")
    print(f"📂 File: {save_path}")
    print("="*50)
    print("ℹ️  NOTA: Questo file contiene l'architettura corretta per Netron.")
    print("    Nel grafico vedrai input 256x256 invece di 224x224.")
    print("    È normale: serve solo per bypassare un bug di visualizzazione della libreria.")

except Exception as e:
    print(f"\n❌ Errore imprevisto: {e}")
    import traceback
    traceback.print_exc()